In [1]:
!pip install gradio -qq
print("Gradio installed successfully!")

Gradio installed successfully!


In [2]:
import joblib
model_filename = '/content/drive/MyDrive/random_forest_model.joblib'
scaler_filename = '/content/drive/MyDrive/scaler.joblib'
joblib.dump(random_search.best_estimator_, model_filename)
print(f"Trained Random Forest model saved to: {model_filename}")
joblib.dump(scaler, scaler_filename)
print(f"StandardScaler saved to: {scaler_filename}")

NameError: name 'random_search' is not defined

In [ ]:
import gradio as gr
import pandas as pd
import joblib
loaded_model = joblib.load('/content/drive/MyDrive/random_forest_model.joblib')
loaded_scaler = joblib.load('/content/drive/MyDrive/scaler.joblib')
def predict_potability(
    ph,
    Hardness,
    Solids,
    Chloramines,
    Sulfate,
    Conductivity,
    Organic_carbon,
    Trihalomethanes,
    Turbidity
):
    input_data = pd.DataFrame([[ph, Hardness, Solids, Chloramines, Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity]],
    columns=['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity'])
    scaled_input = loaded_scaler.transform(input_data)
    prediction = loaded_model.predict(scaled_input)[0]
    prediction_proba = loaded_model.predict_proba(scaled_input)[0]
    output_text = ""
    if prediction == 1:
        output_text += f"The water is **Potable**! (Confidence: {prediction_proba[1]*100:.2f}%)"
    else:
        output_text += f"The water is **NOT Potable**. (Confidence: {prediction_proba[0]*100:.2f}%)\n\n"
        output_text += "**Suggestions to improve potability:**\n"
        output_text += "- **pH Adjustment:** Ensure pH is within the acceptable range (6.5 to 8.5) through aeration or chemical treatment.\n"
        output_text += "- **Hardness Reduction:** Consider water softening methods like ion exchange or reverse osmosis.\n"
        output_text += "- **Solid Removal:** Implement filtration (e.g., sand filtration, membrane filtration) to reduce dissolved solids.\n"
        output_text += "- **Disinfection Optimization:** Monitor chloramine levels carefully to ensure effective disinfection without excess.\n"
        output_text += "- **Sulfate Control:** If sulfate levels are high, consider reverse osmosis or distillation.\n"
        output_text += "- **Organic Carbon Management:** Use activated carbon filtration or advanced oxidation processes.\n"
        output_text += "- **Trihalomethane Reduction:** Improve source water quality, optimize disinfection, or use activated carbon filters.\n"
        output_text += "- **Turbidity Control:** Enhance coagulation, flocculation, sedimentation, and filtration processes."
    return output_text
iface = gr.Interface(
    fn=predict_potability,
    inputs=[
        gr.Slider(minimum=0.0, maximum=14.0, step=0.1, label="pH"),
        gr.Slider(minimum=0.0, maximum=500.0, step=0.1, label="Hardness (mg/L)"),
        gr.Slider(minimum=0.0, maximum=70000.0, step=1.0, label="Solids (ppm)"),
        gr.Slider(minimum=0.0, maximum=15.0, step=0.01, label="Chloramines (ppm)"),
        gr.Slider(minimum=0.0, maximum=500.0, step=0.1, label="Sulfate (mg/L)"),
        gr.Slider(minimum=0.0, maximum=1000.0, step=0.1, label="Conductivity (μS/cm)"),
        gr.Slider(minimum=0.0, maximum=30.0, step=0.01, label="Organic Carbon (ppm)"),
        gr.Slider(minimum=0.0, maximum=150.0, step=0.01, label="Trihalomethanes (μg/L)"),
        gr.Slider(minimum=0.0, maximum=10.0, step=0.01, label="Turbidity (NTU)")
    ],
    outputs=gr.Markdown("text"),
    title="Water Potability Predictor",
    description="Enter the water quality parameters to predict if the water is potable and get suggestions if not."
)
iface.launch(share=True, debug=True)